In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [3]:

# ============================================================
# 0) إعدادات عامة
# ============================================================
DATASET_PATH = "dataset/"  

TRAIN_DIR = os.path.join(DATASET_PATH, "train")
VAL_DIR   = os.path.join(DATASET_PATH, "val")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42


In [4]:

# ============================================================
# 1) تحميل بيانات Train/Val
# ============================================================
print("Train Dir:", TRAIN_DIR)
print("Val Dir  :", VAL_DIR)

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
print("Classes:", class_names)

# حفظ ترتيب الفئات (مهم للتنبؤ الصحيح)
with open("class_names.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(class_names))
print("Saved class names to: class_names.txt")

# تحسين الأداء
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)


Train Dir: dataset/train
Val Dir  : dataset/val
Found 361 files belonging to 2 classes.
Found 91 files belonging to 2 classes.
Classes: ['HAYAWIYA', 'SHAMLAN']
Saved class names to: class_names.txt


In [5]:

# ============================================================
# 2) نموذج Transfer Learning: MobileNetV2 (Feature Extraction)
# ============================================================
base_model = tf.keras.applications.MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=IMG_SIZE + (3,)
)
base_model.trainable = False  # تجميد القاعدة أولاً
base_model.summary()

inputs = layers.Input(shape=IMG_SIZE + (3,))

# preprocess_input الخاص بـ MobileNetV2
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)

x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)

outputs = layers.Dense(len(class_names), activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

Model: "mobilenetv2_1.00_224"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract (Subtract)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [6]:

# ============================================================
# 3) تدريب المرحلة الأولى (Frozen Base)
# ============================================================
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

# ============================================================
# 4) Fine-tuning اختياري (فك آخر طبقات لتحسين التعميم)
# ============================================================
FINE_TUNE = True
FINE_TUNE_LAST_N_LAYERS = 30

if FINE_TUNE:
    base_model.trainable = True
    for layer in base_model.layers[:-FINE_TUNE_LAST_N_LAYERS]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    history_stage2 = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=15,
        initial_epoch=history_stage1.epoch[-1] + 1
    )


Epoch 1/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 12s 664ms/step - accuracy: 0.5069 - loss: 0.9438 - val_accuracy: 0.8352 - val_loss: 0.4274
Epoch 2/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 499ms/step - accuracy: 0.8255 - loss: 0.3763 - val_accuracy: 0.9341 - val_loss: 0.2126
Epoch 3/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 500ms/step - accuracy: 0.9446 - loss: 0.1922 - val_accuracy: 0.9560 - val_loss: 0.1495
Epoch 4/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 6s 486ms/step - accuracy: 0.9640 - loss: 0.1412 - val_accuracy: 0.9560 - val_loss: 0.1236
Epoch 5/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 420ms/step - accuracy: 0.9778 - loss: 0.0926 - val_accuracy: 0.9670 - val_loss: 0.1124
Epoch 6/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 406ms/step - accuracy: 0.9889 - loss: 0.0767 - val_accuracy: 0.9670 - val_loss: 0.0997
Epoch 7/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 402ms/step - accuracy: 0.9861 - loss: 0.0656 - val_accuracy: 0.9670 - val_loss: 0.0961
Epoch 8/20
12/12 ━━━━━━━━━━━━━━━━━━━━ 5s 393ms/step - accuracy: 0.9945 - loss: 0.0550 - val_accuracy: 0

In [7]:

# ============================================================
# 5) حفظ النموذج
# ============================================================
MODEL_PATH = "water_bottle_mobilenetv2.keras"
model.save(MODEL_PATH)
print("Saved model to:", MODEL_PATH)



Saved model to: water_bottle_mobilenetv2.keras


In [ ]:

# ============================================================
# 7) اختبار/تنبؤ بصورة خارج dataset (نفس الملف)
# ============================================================
EXTERNAL_IMAGE_PATH = "images/img.jpg"


def predict_external_image(image_path: str):
    if not os.path.exists(image_path):
        print(f"[WARN] External image not found: {image_path}")
        print("ضع صورة خارج dataset في المسار EXTERNAL_IMAGE_PATH ثم أعد التشغيل.")
        return

    img = tf.keras.preprocessing.image.load_img(image_path, target_size=IMG_SIZE)
    x = tf.keras.preprocessing.image.img_to_array(img)
    x = np.expand_dims(x, axis=0)  # (1, 224, 224, 3)

    # IMPORTANT:
    # preprocess_input موجود داخل النموذج (على inputs)
    # لذلك لا نعمل /255 هنا
    probs = model.predict(x, verbose=0)[0]
    pred_idx = int(np.argmax(probs))


    print("\n=== External Prediction ===")
    print("Image:", image_path)
    print("Prediction:", class_names[pred_idx])
    print("Probabilities:")
    for i, p in enumerate(probs):
        print(f"  {class_names[i]}: {float(p):.4f}")

# نفّذ التنبؤ بعد انتهاء التدريب مباشرة
predict_external_image(EXTERNAL_IMAGE_PATH)



=== External Prediction ===
Image: D:\Year 3\term2\Computer Vesion\project\img.jpg
Prediction: SHAMLAN
Probabilities:
  HAYAWIYA: 0.3229
  SHAMLAN: 0.6771
